# Practice — Core Code Patterns, from One Call to an Agent Loop

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week05/W5_practice_core_patterns.ipynb)

**Goal.** Write, without looking them up, the code patterns every lab is built from: (1) a model call, (2) the message list, (3) a prompt template in a function, (4) JSON output, (5) a tool call: request, run, return, (6) the agent loop, (7) ReAct, and (8) an agent kept in a class. The tools are easy ones: live weather, a calculator, and Wikipedia search.

**Cell labels.** The first line of each code cell says what to do with it:

- `# ✍️ TYPE` — an exercise: write the code between the `FILL IN` lines yourself, without copying from above.
- `# ✍️ WRITE` — answer in words (not checked).
- `# ▶ RUN · example` — the pattern being taught: read each line, then run.
- `# ▶ RUN · setup` or `# ▶ RUN · check` — support cells: just run.

About 85 minutes. Not collected. Needs basic Python: lists, dictionaries, functions, `for` loops, f-strings.

## 0. Setup

*Do:* run the three cells; paste your API key into the second. The third prints `ready`.

In [ ]:
# ▶ RUN · setup: just run
%pip install -q "openai==1.109.1"

In [ ]:
# ▶ RUN · setup: just run
import os

OPENAI_API_KEY = "PASTE-YOUR-KEY-HERE"   # paste your key between the quotes
if OPENAI_API_KEY != "PASTE-YOUR-KEY-HERE":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY   # the client reads the key from here

In [ ]:
# ▶ RUN · setup: just run
import json
from openai import OpenAI

client = OpenAI()                  # the OpenAI library; reads OPENAI_API_KEY from the environment
MODEL = "gpt-4.1"
done = {}                          # the check cells record your results here

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: ready"}],
)
response.choices[0].message.content

## 1. One call and its reply

```python
response = client.chat.completions.create(model=MODEL, messages=messages)
reply = response.choices[0].message.content
```

`messages` is a list of dictionaries; `response` is an object. In `.choices[0].message.content`, `choices` is a list of replies (one by default), `[0]` takes the first, `.message` is that reply, and `.content` is its text.

### 1.1 Look inside the response

*Do:* run the cell. Find the reply text inside the printed message, and the token count in `usage`.

In [ ]:
# ▶ RUN · example: read each line, then run
messages = [{"role": "user", "content": "In one sentence, what is a cancelled order in an online shop?"}]
response = client.chat.completions.create(model=MODEL, messages=messages)

print(type(response))                        # an object
print(response.choices[0].message)           # the whole reply: role, content, tool_calls, ...
print(response.choices[0].message.content)   # only the text
print(response.usage)                        # tokens counted for this call (what you pay for)

### 1.2 ✍️ Write a call

*Do:* the message list is given; write the call and read the reply text into `reply_1`.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
messages = [{"role": "user", "content": "Reply with exactly: PASS"}]
### FILL IN (START) ###
reply_1 = ""        # call the model with messages and read the reply text
### FILL IN (END) ###

In [ ]:
# ▶ RUN · check: just run
done["1.2"] = reply_1 == "PASS"
print(done["1.2"], reply_1)

### 1.3 ✍️ Read another field of the response

*Do:* the call is given; read the call's total token count into `tokens_2` (it is in `response.usage`).

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
messages = [{"role": "user", "content": "In one sentence, what does a shop's order status mean?"}]
response = client.chat.completions.create(model=MODEL, messages=messages)
reply_2 = response.choices[0].message.content
### FILL IN (START) ###
tokens_2 = 0        # read the call's total token count
### FILL IN (END) ###

In [ ]:
# ▶ RUN · check: just run
done["1.3"] = reply_2 != "" and tokens_2 > 0
print(done["1.3"], tokens_2, reply_2)

## 2. The message list

A call sees only the list you send; the model keeps nothing between calls. Each message has a **role** that says who wrote it. The table lists every kind of text a call can carry; the last two appear in Part 5.

| Kind | In the code | Written by | Holds |
|---|---|---|---|
| **System** message | `{"role": "system", "content": ...}`, first in the list | you, once | standing instructions for the whole conversation |
| **User** message | `{"role": "user", "content": ...}` | you, for each request | the question or task, with any data it needs |
| **Assistant** message | `{"role": "assistant", ...}` | the model; you copy it into the list | a reply, or a tool request (Part 5) |
| **Tool** message | `{"role": "tool", "tool_call_id": ..., "content": ...}` | your program (Part 5) | the result of running a requested function |
| **Tool schema** | `tools=[...]`, next to `messages`, not in it | you (Part 5) | the name, purpose, and arguments of a function the model may request |

An **instruction** (text telling the model what to do) applies where you put it: in the system message, to every turn; in a user message, to that request; in a tool schema, whenever the model considers that tool. Names ending in `_SYSTEM` hold system messages; names ending in `_TOOL` hold tool schemas.

```python
messages = [{"role": "system", "content": STAFF_SYSTEM},
            {"role": "user", "content": QUESTION}]
messages.append({"role": "assistant", "content": reply})   # keep the model's reply
messages.append({"role": "user", "content": NEXT_QUESTION})
```

### 2.1 Watch the list grow

`show_messages` prints the list you send: each message's position, role, and text, with `<- NEW` on what was just added. You will use it from here to the end.

*Do:* run the cells one by one. Each cell adds to the list and prints it.

In [ ]:
# ▶ RUN · setup: just run
def show_messages(messages, new_from=0):
    """Print the message list: position, role, and text. Messages at new_from and later are marked NEW."""
    for i in range(len(messages)):
        m = messages[i]
        if "tool_calls" in m:                              # an assistant message that holds requests
            text = ""
            for c in m["tool_calls"]:
                text = text + "request " + c["function"]["name"] + " " + c["function"]["arguments"] + "  "
        elif m["role"] == "tool":
            text = str(json.loads(m["content"])).replace("\n", " ")   # the result, decoded from JSON text
        else:
            text = str(m["content"]).replace("\n", " ")   # keep each message on one line
        mark = ""
        if i >= new_from:
            mark = "   <- NEW"
        print("[" + str(i) + "]", m["role"], "|", text[:110] + mark)
    print()

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 1 - the list we send: a system message and a user message
STAFF_SYSTEM = "You answer questions from shop staff. Answer in at most two sentences."

messages = [
    {"role": "system", "content": STAFF_SYSTEM},              # SYSTEM message: standing instruction
    {"role": "user", "content": "A customer ordered 2 mugs and 1 desk lamp. List the items in the order."},   # USER message
]
show_messages(messages)                                       # 2 messages

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 2 - call the model and keep its reply
response = client.chat.completions.create(model=MODEL, messages=messages)
reply = response.choices[0].message.content
messages.append({"role": "assistant", "content": reply})      # ASSISTANT message
show_messages(messages, new_from=2)                           # 3 messages

### 2.2 Continue the conversation

The follow-up says only "the lamp"; the model can resolve it because the earlier turns are in the list it receives.

*Do:* run the two cells. The reply should list only the 2 mugs. To rerun, start again from 2.1, or the list grows twice.

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 3 - add the follow-up question
messages.append({"role": "user", "content": "The lamp was cancelled. What is left in the order?"})   # USER message
show_messages(messages, new_from=3)                           # 4 messages

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 4 - call the model with the whole list and keep its reply
response = client.chat.completions.create(model=MODEL, messages=messages)
reply = response.choices[0].message.content
messages.append({"role": "assistant", "content": reply})      # ASSISTANT message
show_messages(messages, new_from=4)                           # 5 messages

### 2.3 ✍️ One more turn

*Do:* the question and the call are given; add the reply to the list as an assistant message. The check shows the list: 7 messages, the last two NEW.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
messages.append({"role": "user", "content": "The customer adds 3 notebooks. What is in the order now?"})
response = client.chat.completions.create(model=MODEL, messages=messages)
reply_3 = response.choices[0].message.content
### FILL IN (START) ###
# add reply_3 to messages as an assistant message
### FILL IN (END) ###

In [ ]:
# ▶ RUN · check: just run
show_messages(messages, new_from=5)
roles = []
for m in messages:
    roles.append(m["role"])
done["2.3"] = roles == ["system", "user", "assistant", "user", "assistant", "user", "assistant"] and "notebook" in reply_3.lower() and "mug" in reply_3.lower()
done["2.3"]

## 3. A prompt template inside a function

```python
def ask(prompt):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

text = ask(f"Describe a {product}.")           # f-string: {product} is replaced by its value
review = ask(f"Critique this text:\n{text}")   # the first output is part of the second prompt
```

`ask` sends one **user** message; the instruction and the data travel together in it. A call written many times goes into a function; inside `ask` is the Part 1 call. Later labs write one such function per step or role, for example draft, critique, and revision.

### 3.1 Define `ask` and fill a template

*Do:* run the cell. Change `product` and run it again.

In [ ]:
# ▶ RUN · example: read each line, then run
def ask(prompt):
    """Send one user message and return the reply text."""
    messages = [{"role": "user", "content": prompt}]          # USER message: instruction + data
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

product = "desk lamp"
description = ask(f"Write a one-sentence product description for a {product} sold in an office-supply shop.")
description

### 3.2 Use one output as the next input

The critique prompt contains `description`, returned in 3.1. Triple quotes let a string span several lines.

*Do:* run the cell; the critique should refer to the sentence above.

In [ ]:
# ▶ RUN · example: read each line, then run
critique = ask(f"""List two concrete problems in this product description. Be brief.

Description:
{description}""")
critique

### 3.3 ✍️ Revise with both texts

*Do:* the template has two empty places; put `{description}` and `{critique}` there so the prompt carries both texts.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
### FILL IN (START) ###
# put {description} and {critique} in the two empty places below
prompt = f"""Rewrite the product description to fix the problems listed in the critique.
Return one sentence only.

Description:

Critique:
"""
### FILL IN (END) ###
revised = ask(prompt)

In [ ]:
# ▶ RUN · check: just run
print(revised)
done["3.3"] = description in prompt and critique in prompt
done["3.3"]

## 4. JSON output read into a Python dictionary

```python
response = client.chat.completions.create(model=MODEL, messages=messages,
                                          response_format={"type": "json_object"})
data = json.loads(response.choices[0].message.content)   # JSON text -> dictionary
data["city"]                                             # read one field
```

A reply is always a string. When code needs its parts, ask for JSON and turn the text into a dictionary with `json.loads`. The instruction "return JSON with these keys" goes in the user message; `response_format={"type": "json_object"}` makes the API return valid JSON, and the messages must contain the word "JSON".

### 4.1 Ask for JSON

*Do:* run the cell. The printed type is `str`.

In [ ]:
# ▶ RUN · example: read each line, then run
customer_message = "Hi, I'd like to order 3 notebooks and 2 pen sets, delivered to Busan."

prompt = f"""Extract the order from the customer message.
Return JSON with keys "items" (a list of objects with keys "product" and "quantity") and "city".

Customer message: {customer_message}"""

messages = [{"role": "user", "content": prompt}]          # USER message: instruction + data
response = client.chat.completions.create(model=MODEL, messages=messages,
                                          response_format={"type": "json_object"})   # the API returns valid JSON text
reply = response.choices[0].message.content
print(type(reply))
reply

### 4.2 Read the JSON as a dictionary

*Do:* run the cell. `order["items"]` is a list, so a `for` loop reads each item.

In [ ]:
# ▶ RUN · example: read each line, then run
order = json.loads(reply)        # JSON text -> Python dictionary
print(type(order))
print(order["city"])
for item in order["items"]:      # each item is a dictionary
    print(item["product"], item["quantity"])

### 4.3 ✍️ Another order

*Do:* the prompt and the call are given; turn the reply into a dictionary with `json.loads`, and add each item's quantity inside the loop. Expected: `Seoul`, 5.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
customer_message_2 = "Please send 1 desk lamp and 4 mugs to Seoul."
prompt = f"""Extract the order from the customer message.
Return JSON with keys "items" (a list of objects with keys "product" and "quantity") and "city".

Customer message: {customer_message_2}"""
messages = [{"role": "user", "content": prompt}]
response = client.chat.completions.create(model=MODEL, messages=messages,
                                          response_format={"type": "json_object"})
### FILL IN (START) ###
order_2 = {"items": [], "city": ""}      # turn the reply text into a dictionary
### FILL IN (END) ###
total_quantity = 0
for item in order_2["items"]:
    ### FILL IN (START) ###
    pass                                 # add this item's quantity to total_quantity
    ### FILL IN (END) ###

In [ ]:
# ▶ RUN · check: just run
print(order_2)
done["4.3"] = "city" in order_2 and order_2["city"] == "Seoul" and total_quantity == 5
print(done["4.3"], total_quantity)

## 5. One tool call: request → run → return

A **tool** is a Python function the model may ask your program to run. One tool call takes four steps:

1. **Describe** the function to the model in a **tool schema**.
2. The model returns a **request** (function name and arguments) in an **assistant** message. It runs nothing. Append the message to the list as soon as it arrives.
3. Your code **runs** the function.
4. Your code **returns** the result: append a **tool** message right after the request, and call the model again for the answer.

### 5.1 The function

`get_weather(city)` returns a city's current temperature from Open-Meteo, a free weather service (no key).

*Do:* run the cell.

In [ ]:
# ▶ RUN · example: read each line, then run
import requests

def get_weather(city: str):
    """Get the current temperature of a city in Celsius.

    Args:
        city: City name in English, for example "Seoul".
    """
    try:                                                    # on a network or lookup error, return the error as text
        place = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                             params={"name": city, "count": 1}, timeout=10).json()["results"][0]   # first match
        weather = requests.get("https://api.open-meteo.com/v1/forecast",
                               params={"latitude": place["latitude"], "longitude": place["longitude"],
                                       "current": "temperature_2m"}, timeout=10).json()
        return {"city": place["name"], "temperature_c": weather["current"]["temperature_2m"]}
    except Exception as error:
        return "error: " + str(error)

get_weather("Seoul")

### 5.2 Step 1 — describe it: the tool schema

The model cannot read Python code; it reads a schema, a dictionary with this shape:

```python
{"type": "function",
 "function": {"name": "get_weather",                                   # the function's name
              "description": "Get the current temperature of ...",     # the docstring's first line
              "parameters": {"type": "object",
                             "properties": {"city": {"type": "string",               # from  city: str
                                                     "description": "City name ..."}},  # from the Args: line
                             "required": ["city"]}}}
```

Everything in it comes from the function, so `tool_from_function` builds it for any function written in this docstring format. The schema goes in `tools=[...]`, not in `messages`.

*Do:* run the two cells (the first only defines the helper) and match each entry with the function.

In [ ]:
# ▶ RUN · setup: just run
import inspect

PYTHON_TO_JSON = {str: "string", int: "integer", float: "number", bool: "boolean"}

def tool_from_function(function):
    """Build a tool schema from a function's name, docstring, and type annotations."""
    doc = inspect.getdoc(function)                        # the docstring, indentation removed
    arg_help = {}
    for line in doc.split("\n"):                          # "name: description" lines under Args:
        line = line.strip()
        if ":" in line and not line.endswith(":"):
            name, text = line.split(":", 1)
            arg_help[name.strip()] = text.strip()
    properties = {}
    for name, parameter in inspect.signature(function).parameters.items():   # each parameter and its annotation
        properties[name] = {"type": PYTHON_TO_JSON[parameter.annotation], "description": arg_help[name]}
    return {
        "type": "function",
        "function": {
            "name": function.__name__,                    # the function's own name
            "description": doc.split("\n")[0],            # the docstring's first line
            "parameters": {"type": "object", "properties": properties, "required": list(properties)},
        },
    }

In [ ]:
# ▶ RUN · example: read each line, then run
WEATHER_TOOL = tool_from_function(get_weather)     # TOOL SCHEMA
print(json.dumps(WEATHER_TOOL, indent=2))          # indent=2 prints the dictionary on several lines

### 5.3 Steps 2–4, one cell each

Each cell is one unit of the call and ends by printing the list. `**args` passes a dictionary as keyword arguments: `get_weather(**{"city": "Seoul"})` is `get_weather(city="Seoul")`. `tool_call_id` links a result to its request; the API rejects a tool message with no request before it.

*Do:* run the four cells in order and watch the list grow 2 → 3 → 4 → 5.

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 1 - the list we send
WEATHER_SYSTEM = "You answer questions about the weather. Use the tools for current values; never guess them."

messages = [
    {"role": "system", "content": WEATHER_SYSTEM},                                  # SYSTEM message
    {"role": "user", "content": "What is the temperature in Seoul right now?"},    # USER message
]
show_messages(messages)                                                             # 2 messages

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 2 - call with the tool schema; the reply is a request: check it, then add it
response = client.chat.completions.create(model=MODEL, messages=messages, tools=[WEATHER_TOOL])
message = response.choices[0].message                    # ASSISTANT message
print("content:", message.content)                       # None: no text answer
print("tool_calls:", message.tool_calls)                 # the request

messages.append(message.model_dump(exclude_none=True))   # add it at once, as a dictionary like the others
show_messages(messages, new_from=2)                      # 3 messages

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 3 - run the requested function and add its result
FUNCTIONS = {"get_weather": get_weather}                 # name -> Python function

call = message.tool_calls[0]                             # the request from unit 2
args = json.loads(call.function.arguments)               # JSON text -> dictionary
result = FUNCTIONS[call.function.name](**args)           # get_weather(city="Seoul")

messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})   # TOOL message
show_messages(messages, new_from=3)                      # 4 messages

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 4 - call again with the result in the list; add the answer
response = client.chat.completions.create(model=MODEL, messages=messages, tools=[WEATHER_TOOL])
answer = response.choices[0].message.content
messages.append({"role": "assistant", "content": answer})    # ASSISTANT message
show_messages(messages, new_from=4)                          # 5 messages

### 5.5 ✍️ The same units for Tokyo

*Do:* the three units for `QUESTION_1` are given except the lines that add each message to `messages_1` (and running the request in unit 3). Each check cell prints your list. Afterwards, try a question that needs no tool ("Say hello"): `tool_calls` is `None`, which is how Part 6's loop knows the model has answered.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
## Unit 1-2 - the list, the call, and the request added
QUESTION_1 = "What is the temperature in Tokyo right now?"
messages_1 = [
    {"role": "system", "content": WEATHER_SYSTEM},
    {"role": "user", "content": QUESTION_1},
]
response = client.chat.completions.create(model=MODEL, messages=messages_1, tools=[WEATHER_TOOL])
message_1 = response.choices[0].message
### FILL IN (START) ###
# add message_1 to messages_1 as a dictionary
### FILL IN (END) ###

In [ ]:
# ▶ RUN · check: just run
show_messages(messages_1, new_from=2)                    # expected: 3 messages, the request NEW

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
## Unit 3 - run the request and add the result
call_1 = message_1.tool_calls[0]
args_1 = json.loads(call_1.function.arguments)
### FILL IN (START) ###
result_1 = None     # run the requested function with args_1
# add result_1 as a tool message (tool_call_id = call_1.id)
### FILL IN (END) ###

In [ ]:
# ▶ RUN · check: just run
show_messages(messages_1, new_from=3)                    # expected: 4 messages, the result NEW

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
## Unit 4 - call again and add the answer
response = client.chat.completions.create(model=MODEL, messages=messages_1, tools=[WEATHER_TOOL])
answer_1 = response.choices[0].message.content
### FILL IN (START) ###
# add answer_1 to messages_1 as an assistant message
### FILL IN (END) ###

In [ ]:
# ▶ RUN · check: just run
show_messages(messages_1, new_from=4)                    # expected: 5 messages, the answer NEW
roles = []
for m in messages_1:
    roles.append(m["role"])
found = False
if isinstance(result_1, dict):                           # the tool returned a dictionary
    value = result_1["temperature_c"]
    if str(value) in str(answer_1):
        found = True
    if value == round(value) and str(round(value)) in str(answer_1):   # 21.0 may be written as 21
        found = True
done["5.5"] = roles == ["system", "user", "assistant", "tool", "assistant"] and "Tokyo" in str(result_1) and found
done["5.5"]

## 6. The agent loop

Part 5 was one exchange. Repeat it until a reply has no request, and you have the **agent loop**: the model picks the next action from everything in the list, the program runs it, and the result joins the next call. One reply can hold several requests, so an inner `for` runs each.

```python
messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": QUESTION}]
for step in range(MAX_STEPS):                       # a limit, so the loop always ends
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
    message = response.choices[0].message
    messages.append(message.model_dump(exclude_none=True))       # ASSISTANT message
    if message.tool_calls is None:                  # no request: this reply is the answer
        break
    for call in message.tool_calls:                 # one reply may hold several requests
        args = json.loads(call.function.arguments)
        result = FUNCTIONS[call.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})   # TOOL message
```

### 6.1 Calculator tools

Three small functions, each with a docstring in the 5.2 format, so `tool_from_function` builds their schemas. The system message tells the model to use a tool for every arithmetic step.

*Do:* run the cell; it prints the three tool names.

In [ ]:
# ▶ RUN · example: read each line, then run
def add(a: float, b: float):
    """Add two numbers and return the sum.

    Args:
        a: The first number.
        b: The second number.
    """
    return a + b

def subtract(a: float, b: float):
    """Subtract the second number from the first and return the difference.

    Args:
        a: The number to subtract from.
        b: The number to subtract.
    """
    return a - b

def multiply(a: float, b: float):
    """Multiply two numbers and return the product.

    Args:
        a: The first number.
        b: The second number.
    """
    return a * b

CALC_SYSTEM = "You are a careful calculator. Use the tools for every arithmetic step, one operation per call; never compute in your head."
CALC_TOOLS = [tool_from_function(add), tool_from_function(subtract), tool_from_function(multiply)]   # TOOL SCHEMAS
CALC_FUNCTIONS = {"add": add, "subtract": subtract, "multiply": multiply}

for tool in CALC_TOOLS:
    print(tool["function"]["name"], "-", tool["function"]["description"])

### 6.2 Run the loop

The addition needs the product first, so the model must wait for one result before the next request.

*Do:* run the cell. The list is printed after every step; follow what each step adds. Expected: 4817 × 293 = 1,411,381, then + 1569 = 1,412,950.

In [ ]:
# ▶ RUN · example: read each line, then run
QUESTION = "What is (4817 × 293) + 1569?"
MAX_STEPS = 6

messages = [{"role": "system", "content": CALC_SYSTEM}, {"role": "user", "content": QUESTION}]
show_messages(messages)
for step in range(MAX_STEPS):
    start = len(messages)                                      # where this step's messages begin
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=CALC_TOOLS)
    message = response.choices[0].message
    messages.append(message.model_dump(exclude_none=True))
    if message.tool_calls is None:
        print("step", step, "- the reply has no request: it is the answer")
        show_messages(messages, new_from=start)
        break
    for call in message.tool_calls:
        args = json.loads(call.function.arguments)
        result = CALC_FUNCTIONS[call.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
    print("step", step, "- a request and its result were added")
    show_messages(messages, new_from=start)

### 6.3 ✍️ Write the loop yourself

*Do:* the loop is given except its two appends: the assistant message after each call, and the tool message after each result. Expected: 58 × 94 = 5,452 and 37 × 21 = 777 (these two can come in one reply), then 5,452 + 777 = 6,229.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
QUESTION_2 = "What is (58 × 94) + (37 × 21)?"
messages_2 = [{"role": "system", "content": CALC_SYSTEM}, {"role": "user", "content": QUESTION_2}]
answer_2 = None
for step in range(MAX_STEPS):
    response = client.chat.completions.create(model=MODEL, messages=messages_2, tools=CALC_TOOLS)
    message = response.choices[0].message
    ### FILL IN (START) ###
    # add message to messages_2 as a dictionary
    ### FILL IN (END) ###
    if message.tool_calls is None:
        answer_2 = message.content
        break
    for call in message.tool_calls:
        args = json.loads(call.function.arguments)
        result = CALC_FUNCTIONS[call.function.name](**args)
        ### FILL IN (START) ###
        # add result as a tool message
        ### FILL IN (END) ###

In [ ]:
# ▶ RUN · check: just run
show_messages(messages_2)
done["6.3"] = answer_2 is not None and ("6229" in answer_2 or "6,229" in answer_2)
done["6.3"]

### 6.4 ✍️ Mix two kinds of tools

Registering a tool is now one line per function. Weather and calculator tools together answer a question that needs a lookup and then arithmetic.

*Do:* the loop is given; complete the registration: `TOOLS_3` needs the schemas for `add` and `multiply` too, and `FUNCTIONS_3` their names.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
MIXED_SYSTEM = ("Use get_weather for current temperatures and the calculator tools for every arithmetic step, "
                "one operation per call; never compute in your head.")
QUESTION_3 = "What is the current temperature in Seoul in Fahrenheit? Use F = C × 1.8 + 32."
### FILL IN (START) ###
TOOLS_3 = [tool_from_function(get_weather)]          # add the schemas for add and multiply
FUNCTIONS_3 = {"get_weather": get_weather}           # add their names and functions
### FILL IN (END) ###
messages_3 = [{"role": "system", "content": MIXED_SYSTEM}, {"role": "user", "content": QUESTION_3}]
answer_3 = None
for step in range(MAX_STEPS):
    response = client.chat.completions.create(model=MODEL, messages=messages_3, tools=TOOLS_3)
    message = response.choices[0].message
    messages_3.append(message.model_dump(exclude_none=True))
    if message.tool_calls is None:
        answer_3 = message.content
        break
    for call in message.tool_calls:
        args = json.loads(call.function.arguments)
        result = FUNCTIONS_3[call.function.name](**args)
        messages_3.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

In [ ]:
# ▶ RUN · check: just run
show_messages(messages_3)
requested = []
for m in messages_3:
    if m["role"] == "assistant" and "tool_calls" in m:     # assistant messages that hold requests
        for c in m["tool_calls"]:
            requested.append(c["function"]["name"])
done["6.4"] = "get_weather" in requested and "multiply" in requested and "add" in requested and answer_3 is not None
done["6.4"]

## 7. ReAct: a thought before each action

ReAct has the model write a **Thought** (what the results show, what is missing, why the next action) before each **Action**; the tool's result is the **Observation**.

With function calling, a request carries only a function name and arguments, so there is no place for a thought. The simple fix: give each tool a first argument named `thought`. The model must fill it in every request, so the reason arrives together with the action.

```python
def search(thought: str, query: str):
    """Search Wikipedia ... Args: thought: your reasoning ...  query: search words ..."""
    print("Thought:", thought)          # show the reason when the action runs
    return wikipedia_search(query)
```

The loop is Part 6's loop unchanged. Only two things differ: the tools ask for a thought, and `parallel_tool_calls=False` asks for one request per reply, so each thought is written after the previous observation.

### 7.1 Tools that ask for a thought

*Do:* run the two cells. The first defines a plain Wikipedia search (no key). The second wraps search and weather as tools whose first argument is `thought`, and prints the search tool's schema: `thought` is now its first, required argument.

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 1 - a plain Wikipedia search
import html

def wikipedia_search(query: str):
    """Search Wikipedia and return the top three results: title and a short snippet.

    Args:
        query: Search words, for example "Gyeongbokgung Palace".
    """
    try:
        data = requests.get("https://en.wikipedia.org/w/api.php",
                            params={"action": "query", "list": "search", "srsearch": query,
                                    "srlimit": 3, "format": "json"},
                            headers={"User-Agent": "STML-course-practice/1.0"}, timeout=10).json()
        lines = []
        for item in data["query"]["search"]:
            snippet = item["snippet"].replace('<span class="searchmatch">', "").replace("</span>", "")   # drop highlight tags
            lines.append(item["title"] + " | " + html.unescape(snippet))                               # &#039; -> '
        return "\n".join(lines)
    except Exception as error:                     # a failed search is returned as text, not raised
        return "error: " + str(error)

print(wikipedia_search("Gyeongbokgung Palace"))

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 2 - tools whose first argument is the thought
def search(thought: str, query: str):
    """Search Wikipedia and return the top three results.

    Args:
        thought: Your reasoning: what the results so far show, what is still missing, and why this search.
        query: Search words, for example "Gyeongbokgung Palace".
    """
    print("Thought:", thought)
    return wikipedia_search(query)

def check_weather(thought: str, city: str):
    """Get the current temperature of a city in Celsius.

    Args:
        thought: Your reasoning: what the results so far show, what is still missing, and why this lookup.
        city: City name in English, for example "Seoul".
    """
    print("Thought:", thought)
    return get_weather(city)

REACT_TOOLS = [tool_from_function(search), tool_from_function(check_weather)]   # TOOL SCHEMAS, built as in Part 5
REACT_FUNCTIONS = {"search": search, "check_weather": check_weather}
print(json.dumps(REACT_TOOLS[0], indent=2))

### 7.2 The same loop, one request per reply

*Do:* run the cell. At each step the tool prints the Thought, then the list shows the request (with its thought) and the result. Expected: a search that finds Seoul, then `check_weather` for Seoul.

In [ ]:
# ▶ RUN · example: read each line, then run
TRAVEL_SYSTEM = "You help travellers. Use search to find the city of any place before checking its weather."
QUESTION = "I am visiting Gyeongbokgung Palace this afternoon. What is the temperature there right now?"

messages = [{"role": "system", "content": TRAVEL_SYSTEM}, {"role": "user", "content": QUESTION}]
show_messages(messages)
for step in range(MAX_STEPS):
    start = len(messages)
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=REACT_TOOLS,
                                              parallel_tool_calls=False)   # one request per reply
    message = response.choices[0].message
    messages.append(message.model_dump(exclude_none=True))
    if message.tool_calls is None:
        print("step", step, "- the reply has no request: it is the answer")
        show_messages(messages, new_from=start)
        break
    for call in message.tool_calls:
        args = json.loads(call.function.arguments)
        result = REACT_FUNCTIONS[call.function.name](**args)          # the tool prints its thought
        messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
    print("step", step, "- a request with its thought, and the result, were added")
    show_messages(messages, new_from=start)

### 7.3 ✍️ Read one step

*Do:* choose one result and the Thought after it. In `reading`, write which fact from the result the Thought used, and whether the next action follows from it. (Not checked.)

In [ ]:
# ✍️ WRITE · your answer in words (not checked)
reading = """

"""

## 8. The agent as a class

A loop written in a cell starts a new `messages` list each time, so a follow-up ("And in Busan?") has nothing to refer to. A **class** keeps the list inside an object: every call appends to the same `self.messages`.

```python
class Chat:
    def __init__(self, system):                      # runs once, when the object is created
        self.messages = [{"role": "system", "content": system}]

    def ask(self, question):                         # a method: a function that belongs to the object
        self.messages.append({"role": "user", "content": question})
        ...                                          # call the model with self.messages
        self.messages.append({"role": "assistant", "content": reply})
        return reply
```

`self` is the object itself; `self.messages` is a value stored in it (an **attribute**) that stays between calls.

### 8.1 Messages that accumulate

*Do:* run the cells one by one. Compare with 2.1–2.2: the same list grows, but now `ask` appends for you. The second question says only "the lamp"; it works because the first exchange is still in `chat.messages`.

In [ ]:
# ▶ RUN · example: read each line, then run
class Chat:
    """A conversation that keeps its own message history."""

    def __init__(self, system):
        self.messages = [{"role": "system", "content": system}]               # SYSTEM message, once

    def ask(self, question):
        self.messages.append({"role": "user", "content": question})           # USER message
        response = client.chat.completions.create(model=MODEL, messages=self.messages)
        reply = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": reply})         # ASSISTANT message
        return reply

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 1 - create the object: its list holds only the system message
chat = Chat(STAFF_SYSTEM)
show_messages(chat.messages)                              # 1 message

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 2 - ask: the question and the reply are added by ask
chat.ask("A customer ordered 2 mugs and 1 desk lamp. List the items in the order.")
show_messages(chat.messages, new_from=1)                  # 3 messages

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 3 - a follow-up: the same list grows
chat.ask("The lamp was cancelled. What is left in the order?")
show_messages(chat.messages, new_from=3)                  # 5 messages

### 8.2 ✍️ Write the class yourself

*Do:* complete `MyChat`: the two lines in `ask` that add the question and the reply to `self.messages`, and the body of `reset`. The check asks two connected questions, then calls `reset()`.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
class MyChat:
    def __init__(self, system):
        self.system = system
        self.messages = [{"role": "system", "content": system}]

    def ask(self, question):
        ### FILL IN (START) ###
        # add question to self.messages as a user message
        ### FILL IN (END) ###
        response = client.chat.completions.create(model=MODEL, messages=self.messages)
        reply = response.choices[0].message.content
        ### FILL IN (START) ###
        # add reply to self.messages as an assistant message
        ### FILL IN (END) ###
        return reply

    def reset(self):
        ### FILL IN (START) ###
        pass    # set self.messages back to a list with only the system message
        ### FILL IN (END) ###

In [ ]:
# ▶ RUN · check: just run
my_chat = MyChat("You answer questions from shop staff. Answer in one sentence.")
my_chat.ask("A customer ordered 3 notebooks and 2 pen sets.")
follow_up = my_chat.ask("The customer adds 1 folder. What is in the order now?")
length_before = len(my_chat.messages)
show_messages(my_chat.messages)
my_chat.reset()
print("after reset():")
show_messages(my_chat.messages)
done["8.2"] = length_before == 5 and len(my_chat.messages) == 1 and "notebook" in follow_up.lower() and "folder" in follow_up.lower()
done["8.2"]

### 8.3 The agent loop in a class

`Agent` takes a system message and a **list of functions**. It builds the tool schemas with `tool_from_function` and the name → function dictionary itself, so adding a tool means adding a function to the list. `ask` runs the Part 6 loop on `self.messages` and adds up `response.usage.total_tokens`.

*Do:* run the three cells. The second question only makes sense after the first: its NEW messages sit below the first question's, and the subtraction uses Seoul's temperature from there.

In [ ]:
# ▶ RUN · example: read each line, then run
class Agent:
    """An agent that registers its tools from functions and keeps its history between questions."""

    def __init__(self, system, functions, max_steps=6):
        self.tools = []                                                      # TOOL SCHEMAS
        self.functions = {}                                                  # name -> Python function
        for function in functions:
            self.tools.append(tool_from_function(function))
            self.functions[function.__name__] = function
        self.max_steps = max_steps
        self.messages = [{"role": "system", "content": system}]              # SYSTEM message, once
        self.total_tokens = 0

    def ask(self, question):
        self.messages.append({"role": "user", "content": question})          # USER message
        for step in range(self.max_steps):
            response = client.chat.completions.create(model=MODEL, messages=self.messages, tools=self.tools)
            self.total_tokens = self.total_tokens + response.usage.total_tokens
            message = response.choices[0].message
            self.messages.append(message.model_dump(exclude_none=True))      # ASSISTANT message
            if message.tool_calls is None:
                return message.content
            for call in message.tool_calls:
                args = json.loads(call.function.arguments)
                result = self.functions[call.function.name](**args)
                self.messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})   # TOOL message
        return None                                                          # the step limit ended the loop

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 1 - create the agent and ask the first question
weather = Agent(MIXED_SYSTEM, [get_weather, add, subtract, multiply])
weather.ask("What is the temperature in Seoul right now?")
show_messages(weather.messages, new_from=1)               # user, request, result, answer

In [ ]:
# ▶ RUN · example: read each line, then run
## Unit 2 - a follow-up that needs the first answer
start = len(weather.messages)                             # where the new messages will begin
weather.ask("And in Busan? How many degrees warmer or colder is Busan than Seoul?")
show_messages(weather.messages, new_from=start)
print("tokens so far:", weather.total_tokens)

### 8.4 ✍️ A second object with its own history

*Do:* complete the line that creates `converter`: an `Agent` with `CONVERTER_SYSTEM` and the functions `add` and `multiply`. The two questions are given. Expected: 86 and 53.6. The check also confirms that `weather` did not receive these messages.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
CONVERTER_SYSTEM = ("You convert Celsius to Fahrenheit with F = C × 1.8 + 32. "
                    "Use the calculator tools for every arithmetic step, one operation per call.")
### FILL IN (START) ###
converter = Agent(CONVERTER_SYSTEM, [add])      # give it both calculator functions it needs: add and multiply
### FILL IN (END) ###
answer_30 = converter.ask("Convert 30 °C to Fahrenheit.")
answer_12 = converter.ask("And 12 °C?")

In [ ]:
# ▶ RUN · check: just run
weather_questions = 0
for m in weather.messages:
    if m["role"] == "user":
        weather_questions = weather_questions + 1
show_messages(converter.messages)                    # the converter's own list
print("user messages in weather:", weather_questions)
done["8.4"] = "multiply" in converter.functions and "86" in str(answer_30) and "53.6" in str(answer_12) and weather_questions == 2
done["8.4"]

### 8.5 ✍️ Two objects working together

An advisor answers, a reviewer comments, the advisor revises. The reviewer's **system** message is `REVIEW_SYSTEM`; the answer reaches it in a **user** message, and the review goes back to the advisor in another **user** message. The advisor keeps its history, so "revise your answer" is enough.

*Do:* the two objects and the first answer are given; write the two `ask` lines: the reviewer receives the question and the answer, and the advisor receives the review.

In [ ]:
# ✍️ TYPE · exercise: write the code between the FILL IN lines yourself
REVIEW_SYSTEM = ("You review weather advice. Point out anything missing, such as the actual temperature "
                 "or a clear recommendation. Answer in at most three sentences.")
QUESTION_4 = "Should I take a jacket for a walk in Seoul right now?"
advisor = Agent(WEATHER_SYSTEM, [get_weather])
reviewer = Chat(REVIEW_SYSTEM)
answer = advisor.ask(QUESTION_4)
### FILL IN (START) ###
review = ""     # ask the reviewer with a message that holds QUESTION_4 and answer
revised = ""    # ask the advisor to revise, with a message that holds review
### FILL IN (END) ###

In [ ]:
# ▶ RUN · check: just run
show_messages(reviewer.messages)                     # the answer reached the reviewer as a USER message
print("REVISED:", revised)
advisor_questions = 0
for m in advisor.messages:
    if m["role"] == "user":
        advisor_questions = advisor_questions + 1
done["8.5"] = advisor_questions == 2 and len(reviewer.messages) == 3 and revised != ""
done["8.5"]

## Summary

*Do:* run the cell; redo any `False` item.

In [ ]:
# ▶ RUN · setup: just run
for key in done:
    print(key, done[key])